# Задача 3 – Создание хеш-таблицы

## Код

In [1]:
class HashTable:
    def __init__(self, capacity=8):
        if type(capacity) is not int or capacity <= 0:
            raise ValueError('Размер таблицы должен быть положительным целым числом')
        self.buckets = [[] for _ in range(capacity)]
        self.size = 0

    def _index(self, key):
        return hash(key) % len(self.buckets)

    def insert(self, key, value):
        index = self._index(key)
        for pair in self.buckets[index]:
            if pair[0] == key:
                pair[1] = value
                return

        # расширяем таблицу, если после вставки заполненность превысит 0.75
        if (self.size + 1) / len(self.buckets) > 0.75:
            self._resize()
            index = self._index(key)

        self.buckets[index].append([key, value])
        self.size += 1

    def get(self, key):
        index = self._index(key)
        for pair in self.buckets[index]:
            if pair[0] == key:
                return pair[1]
        raise KeyError(key)

    def delete(self, key):
        index = self._index(key)
        for i in range(len(self.buckets[index])):
            if self.buckets[index][i][0] == key:
                self.buckets[index].pop(i)
                self.size -= 1
                return
        raise KeyError(key)

    def _resize(self):
        old_buckets = self.buckets
        self.buckets = [[] for _ in range(len(old_buckets) * 2)]
        for bucket in old_buckets:
            for pair in bucket:
                index = self._index(pair[0])
                self.buckets[index].append(pair)

    def __len__(self):
        return self.size

## Идея

#### Хранение данных

В классе HashTable заводим список buckets, каждый элемент которого тоже будет списком. Эти списки будем называть корзинами, в них записываем пары [key, value]. Таким образом, для хранения используем только list

Номер корзины получаем с помощью hash(key) % len(buckets). Функцию hash используем готовую, остальные операции реализуем сами. Ключ должен быть хешируемым, например числом или строкой. Сам ключ и его хеш не должны меняться, пока пара лежит в таблице

Отдельно заводим size, в котором считаем количество пар. Через len(table) возвращаем size

#### Коллизии

Разные ключи могут попасть в одну корзину. Например, при размере таблицы 4 ключи 1, 5 и 9 попадут в корзину с индексом 1. В таком случае записываем все пары в один список, а при поиске сравниваем сами ключи, а не только их хеши

#### Вставка

Сначала получаем индекс корзины и проверяем, есть ли в ней такой ключ. Если есть, перезаписываем значение, количество пар не меняется. Если ключа нет, добавляем новую пару и увеличиваем size на 1

#### Поиск и удаление

В get идем по корзине и, если нашли нужный ключ, возвращаем значение пары. В delete так же находим пару, удаляем ее через pop и уменьшаем size на 1

Если ключа нет, вызываем KeyError. Это позволяет отличить отсутствие ключа от случая, когда для него записано значение None. Если передать нехешируемый ключ, например список, получим TypeError

#### Расширение таблицы

Перед добавлением нового ключа проверяем, не станет ли отношение числа пар к числу корзин больше 0.75. Если станет, создаем список с количеством корзин в два раза больше и переносим туда все пары

При переносе для каждой пары заново считаем индекс, тк изменился делитель в остатке от деления. Например, ключ 5 при размере 4 лежал в корзине 1, а при размере 8 попадет в корзину 5. Сам size при переносе не меняем

После удаления таблицу не уменьшаем. Расширение помогает распределить пары по корзинам, но если у разных ключей полностью одинаковые хеши, они все равно останутся в одной корзине

Асимптотическая сложность поиска и удаления: **O(1)** в среднем. Вставка тоже занимает **O(1)** в среднем амортизированно, то есть с учетом редких расширений на длинной последовательности вставок

В худшем случае, когда все ключи попадают в одну корзину, операции занимают **O(n)**, где n — количество пар. Отдельное расширение занимает **O(n + c)**, где c — число корзин. Операция len(table) занимает **O(1)**

Объем потребляемой памяти: **O(n + c)**. После удаления число корзин не уменьшается, поэтому память зависит и от предыдущего размера таблицы

В этих оценках считаем, что хеширование и сравнение ключей занимают O(1)

## Визуализация

![Схема](hash_table_schema.png)

## Тесты

In [2]:
def test_hash_table():
    table = HashTable()
    assert len(table) == 0

    # добавление и поиск
    table.insert('cat', 5)
    table.insert('dog', 7)
    assert table.get('cat') == 5
    assert table.get('dog') == 7
    assert len(table) == 2

    # повторный ключ обновляет значение, а не добавляет новую пару
    table.insert('cat', 10)
    assert table.get('cat') == 10
    assert len(table) == 2

    # None можно хранить как значение
    table.insert('empty', None)
    assert table.get('empty') is None

    table.delete('cat')
    assert len(table) == 2
    assert table.get('dog') == 7
    for operation in (table.get, table.delete):
        try:
            operation('cat')
        except KeyError:
            pass
        else:
            assert False, 'Для отсутствующего ключа ожидаем KeyError'

    # удаленный ключ можно добавить заново
    table.insert('cat', 20)
    assert table.get('cat') == 20
    print('OK')


def test_collisions():
    # при размере 8 все три ключа попадают в корзину 1
    table = HashTable(8)
    table.insert(1, 'a')
    table.insert(9, 'b')
    table.insert(17, 'c')
    assert len(table.buckets[1]) == 3
    assert table.get(1) == 'a'
    assert table.get(9) == 'b'
    assert table.get(17) == 'c'

    # обновление и удаление из середины цепочки
    table.insert(9, 'new')
    assert table.get(9) == 'new'
    assert len(table) == 3
    table.delete(9)
    assert table.get(1) == 'a'
    assert table.get(17) == 'c'
    table.delete(1)
    assert table.get(17) == 'c'
    table.delete(17)
    assert table.buckets[1] == []
    assert len(table) == 0
    print('OK')


def test_resize():
    table = HashTable(4)
    for i in range(3):
        table.insert(i, i * 10)
    assert len(table.buckets) == 4

    # обновление существующего ключа не вызывает расширение
    table.insert(1, 100)
    assert len(table.buckets) == 4
    table.insert(3, 30)
    assert len(table.buckets) == 8
    assert len(table) == 4
    assert table.get(1) == 100

    # несколько расширений подряд, потом удаление и повторная вставка
    table = HashTable(1)
    for i in range(200):
        table.insert(i, i * 10)
    assert len(table) == 200
    assert len(table) / len(table.buckets) <= 0.75
    for i in range(200):
        assert table.get(i) == i * 10
    for i in range(0, 200, 2):
        table.delete(i)
    assert len(table) == 100
    for i in range(1, 200, 2):
        assert table.get(i) == i * 10
    for i in range(0, 200, 2):
        table.insert(i, -i)
    assert len(table) == 200
    for i in range(200):
        assert table.get(i) == (-i if i % 2 == 0 else i * 10)

    # внутри таблицы только списки корзин и списки пар
    assert type(table.buckets) is list
    for bucket in table.buckets:
        assert type(bucket) is list
        for pair in bucket:
            assert type(pair) is list
            assert len(pair) == 2
    print('OK')


def test_edge_cases():
    for capacity in (0, -1, 2.5, True):
        try:
            HashTable(capacity)
        except ValueError:
            pass
        else:
            assert False, 'Ожидаем ошибку размера'

    table = HashTable()
    # пустая строка, отрицательный ключ и None
    for key in ('', -7, None):
        table.insert(key, 42)
        assert table.get(key) == 42

    # равные ключи считаем одним ключом
    table.insert(1, 'first')
    size = len(table)
    table.insert(True, 'updated')
    assert len(table) == size
    assert table.get(1) == 'updated'

    # нехешируемый ключ отклоняется без изменения таблицы
    size = len(table)
    for operation in (lambda key: table.insert(key, 1), table.get, table.delete):
        try:
            operation([])
        except TypeError:
            pass
        else:
            assert False, 'Список нельзя использовать как ключ'
    assert len(table) == size
    print('OK')


def test_same_hash():
    # разные ключи с одинаковым хешем, в том числе после расширений
    class Key:
        def __init__(self, val):
            self.val = val

        def __hash__(self):
            return 1

        def __eq__(self, other):
            return isinstance(other, Key) and self.val == other.val

    table = HashTable(2)
    for i in range(30):
        table.insert(Key(i), i)
    assert len(table) == 30
    for i in range(30):
        assert table.get(Key(i)) == i
    table.insert(Key(15), 100)
    assert len(table) == 30
    assert table.get(Key(15)) == 100
    table.delete(Key(15))
    assert len(table) == 29
    for i in range(30):
        if i != 15:
            assert table.get(Key(i)) == i
    try:
        table.get(Key(15))
    except KeyError:
        pass
    else:
        assert False, 'Удаленный ключ не должен находиться'
    print('OK')

In [3]:
test_hash_table()
test_collisions()
test_resize()
test_edge_cases()
test_same_hash()

OK
OK
OK
OK
OK
